# Parcial 1: Teoría de Aprendizaje de Máquina - 2026-2

**Profesor:** Diego Armando Pérez Rosero

**Tema:** Métodos kernel, espacios de características y formulación matricial

**Presentado por:** Marlyn Nathalia Mora Riascos y Hernan Jair Telpiz Cuaran

# **1.5**

In [1]:
import matplotlib.pyplot as plt
import numpy as np

In [2]:
#1. PARAMETROS

np.random.seed(42) #semilla aleatoria

N = 300                     # numero total de observaciones (filas)
P = 5                       # numero total de variables (columnas)
Q = 20                      # numero total de caracteristicas expandidas
X = np.random.randn(N, P)   # Matriz de N filas y P columnas N(0, 1)

#RUIDO

sigma_n = 0.5               # desviacion estandar
sigma_n_sq = sigma_n**2     # varianza n

# REGULARIZACION Y PRIOR

lam = 1.0                         #lamda
sigma_w_sq = sigma_n_sq / lam     #varianza w


In [5]:
#2. Matriz de diseño Phi(X) de Q = 20 características


columnas_originales = X                  # (300, 5)
columnas_cuadradas = X **2               # (300, 5)

pares = []                               # Acumulador de parejas

for i in range (5):                      # i va de 0 a 4
  for j in range(i+1, 5):                # j siempre arranca en i + 1si

    pares.append(X[:, i] * X[:, j])      # Multiplica la columna i con lcolumna
    print(X[:, i] * X[:, j])
productos_pares=np.column_stack(pares)   # (300, 10)

#Concatenacion Final
Phi = np.hstack([columnas_originales, columnas_cuadradas, productos_pares])

print("Dimensiones de Phi:", Phi.shape)


[-6.86778352e-02 -3.69752083e-01  2.15827408e-01  5.69502308e-01
 -3.30908757e-01 -1.27671188e-01 -1.11452803e+00 -2.54989794e-01
  1.26549749e-01  3.31588151e-01 -1.24798994e-01 -7.81546595e-01
  8.89629986e-02 -9.76630093e-02  5.55839656e-01  7.15442034e-02
 -7.84475928e-02 -4.59309460e-01  9.40336819e-02 -4.33376452e-01
  5.95369083e-01  7.62115029e-01  5.08740610e-02 -1.04672419e-02
 -7.19354529e-01 -2.16972587e+00 -1.06318098e-01 -1.21399123e+00
  2.97322605e-01 -9.67076357e-01  8.67827995e-02 -1.33281867e+00
 -7.67156936e-01  3.39330755e-01  7.25675020e-01  1.07549472e-02
 -5.36295495e-01  3.37891898e-01 -3.82394867e-01 -3.40565631e-01
  2.00641615e-01 -4.83012970e-01  6.48283648e-01 -5.86550551e-01
 -4.32208137e+00  6.99977652e-02 -1.58094131e-01 -1.28377641e+00
  9.09310093e-02  1.58869794e-01 -1.15731740e+00 -6.13481546e-01
  5.51758664e-02 -2.03689537e+00 -2.06946977e+00  9.19861769e-02
  7.51633164e-02  2.96297832e-01  1.02604464e-01 -6.23341386e-01
  4.64387288e-01  1.49905

In [6]:
#3. Generación de pesos w* , ruido (eta) y t.

#Son Q = 20 coeficientes aleatorios con distribución normal estándar N(0, 1)
w_star = np.random.randn(Q)             # (20, )


#Sigue una distribución normal con media 0 y desviación estándar 0.5
eta = np.random.normal(0, sigma_n, N)   # (300, )

t = Phi @ w_star + eta                  # Vector de respuestas: (300,)

print("Dimensiones de w*:", w_star.shape)   # (20,)
print("Dimensiones de t:", t.shape)         # (300,)


Dimensiones de w*: (20,)
Dimensiones de t: (300,)


In [7]:
# 4A. Partición Train (80%) y Test (20%)

corte = int(0.8 * N)  # Punto de corte al 80% (240 muestras)

# Separación de características
Phi_train = Phi[:corte] #Primeras 240 filas para entrenar (240, 20)

Phi_test = Phi[corte:]  #Últimas 60 filas para probar (60, 20)


# Separación de respuestas
t_train = t[:corte]  # Vector de respuestas entrenamient(240, )
t_test = t[corte:]   # Vector de respuestas para prueba (60, )

In [8]:
# 4B. AJUSTE DE LOS CUATRO ESTIMADORES (SOLO CON TRAIN)


identidad = np.eye(Q)  # Matriz identidad de dimensión 20x20

# 1. Mínimos Cuadrados Ordinarios (OLS): (Phi^T @ Phi)^(-1) @ (Phi^T @ t)      a.w =b
w_ols = np.linalg.solve(Phi_train.T @ Phi_train, Phi_train.T @ t_train)

# 2. Máxima Verosimilitud (MLE): idéntica ecuación normal bajo ruido gaussiano
w_mle = np.linalg.solve(Phi_train.T @ Phi_train, Phi_train.T @ t_train)

# 3. Ridge: agrega regularización cuadrática lambda en la diagonal
w_ridge = np.linalg.solve(
    Phi_train.T @ Phi_train + lam * identidad, Phi_train.T @ t_train
)

# 4. MAP: penalización probabilística por cociente de varianzas en la diagonal
w_map = np.linalg.solve(
    Phi_train.T @ Phi_train + (sigma_n_sq / sigma_w_sq) * identidad,
    Phi_train.T @ t_train,
)

# Comprobación de coincidencias teóricas
print(
    "Diferencia máxima |OLS - MLE|:  ", np.max(np.abs(w_ols - w_mle))
)
print(
    "Diferencia máxima |Ridge - MAP|:", np.max(np.abs(w_ridge - w_map))
)

Diferencia máxima |OLS - MLE|:   0.0
Diferencia máxima |Ridge - MAP|: 0.0


In [9]:
# 5. REPORTE DE MÉTRICAS (MSE TRAIN, MSE TEST Y ||w - w*||_2)

modelos = [("OLS", w_ols), ("MLE", w_mle), ("Ridge", w_ridge), ("MAP", w_map)]

print(f"{'Modelo':<8} | {'MSE Train':<12} | {'MSE Test':<12} | {'||w - w*||':<12}")
print("-" * 52)

for nombre, w in modelos:
    # 1. Error cuadrático medio en entrenamiento
    mse_train = np.mean((t_train - Phi_train @ w) ** 2)

    # 2. Error cuadrático medio en prueba (generalización)
    mse_test = np.mean((t_test - Phi_test @ w) ** 2)

    # 3. Distancia euclidiana respecto a los parámetros reales
    error_w = np.linalg.norm(w - w_star)

    print(
        f"{nombre:<8} | {mse_train:<12.5f} | {mse_test:<12.5f} | {error_w:<12.5f}"
    )

Modelo   | MSE Train    | MSE Test     | ||w - w*||  
----------------------------------------------------
OLS      | 0.23097      | 0.24823      | 0.10377     
MLE      | 0.23097      | 0.24823      | 0.10377     
Ridge    | 0.23108      | 0.24942      | 0.10394     
MAP      | 0.23108      | 0.24942      | 0.10394     


# **2.4 Implementación con MNIST y Fashion-MNIST (MMD²)**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Semilla fija para reproducibilidad
np.random.seed(42)


In [ ]:
# 1. Carga y preprocesamiento de datasets (MNIST y Fashion-MNIST)
try:
    from tensorflow.keras.datasets import mnist, fashion_mnist
    (X_mnist_raw, y_mnist), _ = mnist.load_data()
    (X_fashion_raw, y_fashion), _ = fashion_mnist.load_data()
except Exception:
    import os
    os.environ["KERAS_BACKEND"] = "torch"
    from keras.datasets import mnist, fashion_mnist
    (X_mnist_raw, y_mnist), _ = mnist.load_data()
    (X_fashion_raw, y_fashion), _ = fashion_mnist.load_data()

# Normalización a [0, 1] y aplanado a vectores de 784 píxeles (28x28)
X_mnist = X_mnist_raw.reshape(-1, 784) / 255.0
X_fashion = X_fashion_raw.reshape(-1, 784) / 255.0

print("Datasets cargados y normalizados:")
print(f"MNIST:         {X_mnist.shape}, etiquetas: {y_mnist.shape}")
print(f"Fashion-MNIST: {X_fashion.shape}, etiquetas: {y_fashion.shape}")


In [ ]:
# 2. Partición en 2 grupos:
# - MNIST: números con números (dígitos 0 al 4 vs dígitos 5 al 9)
# - Fashion-MNIST: prendas entre sí (clases 0 al 4 vs clases 5 al 9)
n_samples = 500

# MNIST: Grupo 1 (dígitos 0 a 4) y Grupo 2 (dígitos 5 a 9)
idx_m1 = np.where(y_mnist <= 4)[0]
idx_m2 = np.where(y_mnist >= 5)[0]
np.random.shuffle(idx_m1)
np.random.shuffle(idx_m2)

MNIST_04 = X_mnist[idx_m1[:n_samples]]   # Dígitos 0-4 (n = 500)
MNIST_59 = X_mnist[idx_m2[:n_samples]]   # Dígitos 5-9 (m = 500)

# Fashion-MNIST: Grupo 1 (clases 0 a 4) y Grupo 2 (clases 5 a 9)
idx_f1 = np.where(y_fashion <= 4)[0]
idx_f2 = np.where(y_fashion >= 5)[0]
np.random.shuffle(idx_f1)
np.random.shuffle(idx_f2)

FASHION_04 = X_fashion[idx_f1[:n_samples]]  # Clases 0-4 (tops/prendas, n = 500)
FASHION_59 = X_fashion[idx_f2[:n_samples]]  # Clases 5-9 (calzado/accesorios, m = 500)

print(f"MNIST Grupo 1 (dígitos 0-4):      {MNIST_04.shape}")
print(f"MNIST Grupo 2 (dígitos 5-9):      {MNIST_59.shape}")
print(f"Fashion Grupo 1 (clases 0-4):     {FASHION_04.shape}")
print(f"Fashion Grupo 2 (clases 5-9):     {FASHION_59.shape}")


In [ ]:
# 3. Calibración de sigma mediante la heurística de la mediana (muestra combinada)
muestra_combinada = np.vstack([MNIST_04, MNIST_59])
n_comb = len(muestra_combinada)

# Distancias euclidianas al cuadrado: ||u - v||^2 = ||u||^2 + ||v||^2 - 2<u, v>
normas_sq = np.sum(muestra_combinada**2, axis=1, keepdims=True)
dists_sq = np.maximum(normas_sq + normas_sq.T - 2 * (muestra_combinada @ muestra_combinada.T), 0.0)

# Pares únicos fuera de la diagonal
dists_pares = np.sqrt(dists_sq[np.triu_indices(n_comb, k=1)])
sigma = np.median(dists_pares)

print(f"Ancho de banda calibrado por mediana (sigma): {sigma:.4f}")


In [ ]:
# 4. Función de Kernel Gaussiano (RBF)
def rbf_kernel(X1, X2, sigma):
    sq1 = np.sum(X1**2, axis=1, keepdims=True)
    sq2 = np.sum(X2**2, axis=1, keepdims=True)
    sq_dist = np.maximum(sq1 + sq2.T - 2 * (X1 @ X2.T), 0.0)
    return np.exp(-sq_dist / (2.0 * (sigma**2)))


In [ ]:
# 5. Estimador Empírico Matricial de MMD^2 (según fórmula 2.3 del parcial)
def calcular_mmd2(X, Y, sigma):
    n = X.shape[0]
    m = Y.shape[0]
    
    # Matrices de Gram
    K_XX = rbf_kernel(X, X, sigma)
    K_YY = rbf_kernel(Y, Y, sigma)
    K_XY = rbf_kernel(X, Y, sigma)
    
    # Vectores de unos para la formulación matricial pura
    ones_n = np.ones((n, 1))
    ones_m = np.ones((m, 1))
    
    # Términos del estimador sesgado
    term_XX = (1.0 / (n**2)) * (ones_n.T @ K_XX @ ones_n)[0, 0]
    term_YY = (1.0 / (m**2)) * (ones_m.T @ K_YY @ ones_m)[0, 0]
    term_XY = (2.0 / (n * m)) * (ones_n.T @ K_XY @ ones_m)[0, 0]
    
    mmd2 = term_XX + term_YY - term_XY
    return mmd2, term_XX, term_YY, term_XY, K_XX, K_YY, K_XY


$$\widehat{\mathrm{MMD}}_b^2 = \frac{1}{n^2} \mathbf{1}_n^\top K_{XX} \mathbf{1}_n + \frac{1}{m^2} \mathbf{1}_m^\top K_{YY} \mathbf{1}_m - \frac{2}{nm} \mathbf{1}_n^\top K_{XY} \mathbf{1}_m$$


In [ ]:
# 6. Cálculo numérico de MMD^2:
# 6.1 Comparaciones globales (n = m = 500) según particiones solicitadas
mmd_mnist, tXX_m, tYY_m, tXY_m, K_m_XX, K_m_YY, K_m_XY = calcular_mmd2(MNIST_04, MNIST_59, sigma)
mmd_fashion, tXX_f, tYY_f, tXY_f, K_f_XX, K_f_YY, K_f_XY = calcular_mmd2(FASHION_04, FASHION_59, sigma)
mmd_cross, tXX_c, tYY_c, tXY_c, _, _, K_cross = calcular_mmd2(MNIST_04, FASHION_04, sigma)

print("=" * 65)
print(" RESULTADOS NUMÉRICOS DE MMD^2 (PARTICIONES GLOBALES)")
print("=" * 65)
print(f"1. MNIST (Dígitos 0-4 vs 5-9):        MMD^2 = {mmd_mnist:.6f}")
print(f"2. Fashion-MNIST (Clases 0-4 vs 5-9):  MMD^2 = {mmd_fashion:.6f}")
print(f"3. MNIST vs. Fashion-MNIST (Cruzado):  MMD^2 = {mmd_cross:.6f}")

# 6.2 Cálculo de matrices de calor cruzadas 5x5 (Grupo 0-4 en filas vs Grupo 5-9 en columnas)
muestras_mnist = [X_mnist[y_mnist == d][:100] for d in range(10)]
muestras_fashion = [X_fashion[y_fashion == d][:100] for d in range(10)]

def matriz_mmd_cruzada_5x5(muestras_por_clase, s):
    M = np.zeros((5, 5))
    # Auto-términos de filas (clases 0 a 4) y columnas (clases 5 a 9)
    terminos_filas = [np.mean(rbf_kernel(muestras_por_clase[i], muestras_por_clase[i], s)) for i in range(5)]
    terminos_cols = [np.mean(rbf_kernel(muestras_por_clase[j + 5], muestras_por_clase[j + 5], s)) for j in range(5)]
    for i in range(5):
        for j in range(5):
            k_cruzado = np.mean(rbf_kernel(muestras_por_clase[i], muestras_por_clase[j + 5], s))
            M[i, j] = max(terminos_filas[i] + terminos_cols[j] - 2 * k_cruzado, 0.0)
    return M

matriz_mnist_5x5 = matriz_mmd_cruzada_5x5(muestras_mnist, sigma)
matriz_fashion_5x5 = matriz_mmd_cruzada_5x5(muestras_fashion, sigma)
print("\nMatrices de calor cruzadas 5x5 (0-4 vs 5-9) calculadas con éxito.")


In [ ]:
# 7. Visualización: Discrepancia global y Mapas de Calor cruzados (0 al 4 vs 5 al 9)
fig = plt.figure(figsize=(18, 5.5))

# 1. Gráfico de barras de MMD^2 entre grupos
ax1 = fig.add_subplot(1, 3, 1)
casos = ["MNIST\n(0-4 vs 5-9)", "Fashion-MNIST\n(0-4 vs 5-9)", "MNIST vs\nFashion"]
valores = [mmd_mnist, mmd_fashion, mmd_cross]
colores = ["#2b5c8f", "#2a9d8f", "#e76f51"]
barras = ax1.bar(casos, valores, color=colores, width=0.55, edgecolor="black")
ax1.set_title(r"Discrepancia Global $\widehat{\mathrm{MMD}}_b^2$", fontsize=12, fontweight="bold")
ax1.set_ylabel(r"$\widehat{\mathrm{MMD}}_b^2$")
ax1.grid(True, linestyle="--", alpha=0.5)
for bar in barras:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2, h + 0.005, f"{h:.4f}", ha="center", fontsize=10, fontweight="bold")

# 2. Mapa de calor cruzado: MNIST (Dígitos 0-4 en Y vs Dígitos 5-9 en X)
ax2 = fig.add_subplot(1, 3, 2)
im1 = ax2.imshow(matriz_mnist_5x5, cmap="viridis")
ax2.set_title(r"MNIST: Discrepancia $\widehat{\mathrm{MMD}}^2$" + "\n(Dígitos 0-4 vs Dígitos 5-9)", fontsize=11, fontweight="bold")
ax2.set_xlabel("Dígitos Grupo 2 (5 al 9)", fontweight="bold")
ax2.set_ylabel("Dígitos Grupo 1 (0 al 4)", fontweight="bold")
ax2.set_xticks(range(5))
ax2.set_yticks(range(5))
ax2.set_xticklabels([5, 6, 7, 8, 9])
ax2.set_yticklabels([0, 1, 2, 3, 4])
for i in range(5):
    for j in range(5):
        val = matriz_mnist_5x5[i, j]
        color_texto = "white" if val < matriz_mnist_5x5.max() * 0.6 else "black"
        ax2.text(j, i, f"{val:.3f}", ha="center", va="center", color=color_texto, fontsize=9)
plt.colorbar(im1, ax=ax2, fraction=0.046, pad=0.04)

# 3. Mapa de calor cruzado: Fashion-MNIST (Prendas 0-4 en Y vs Prendas 5-9 en X)
etiquetas_f_y = ["Camiseta (0)", "Pantalón (1)", "Suéter (2)", "Vestido (3)", "Abrigo (4)"]
etiquetas_f_x = ["Sandalia (5)", "Camisa (6)", "Zapatilla (7)", "Bolso (8)", "Botín (9)"]

ax3 = fig.add_subplot(1, 3, 3)
im2 = ax3.imshow(matriz_fashion_5x5, cmap="viridis")
ax3.set_title(r"Fashion-MNIST: Discrepancia $\widehat{\mathrm{MMD}}^2$" + "\n(Prendas 0-4 vs Prendas 5-9)", fontsize=11, fontweight="bold")
ax3.set_xlabel("Grupo 2 (Calzado y Accesorios: 5 al 9)", fontweight="bold")
ax3.set_ylabel("Grupo 1 (Prendas y Tops: 0 al 4)", fontweight="bold")
ax3.set_xticks(range(5))
ax3.set_yticks(range(5))
ax3.set_xticklabels(etiquetas_f_x, rotation=35, ha="right", fontsize=8.5)
ax3.set_yticklabels(etiquetas_f_y, fontsize=8.5)
for i in range(5):
    for j in range(5):
        val = matriz_fashion_5x5[i, j]
        color_texto = "white" if val < matriz_fashion_5x5.max() * 0.6 else "black"
        ax3.text(j, i, f"{val:.3f}", ha="center", va="center", color=color_texto, fontsize=8.5)
plt.colorbar(im2, ax=ax3, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()
